### Step 1: Source Data Partitioning & Generation
* Reads raw YouTube trending data from the Unity Catalog Volume.
* Repartitions the dataset into 1,000 JSON files to simulate an ongoing file-arrival stream in ADLS Gen2.

In [0]:
source_csv_path = "/Volumes/dbr_dev_ua5816bd/natalkamartinuk55/raw_data/USvideos.csv"

landing_path = "abfss://natalkamartinuk55@dlsua5816bd.dfs.core.windows.net/autoloader_source/initial_files"

df_source = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", "\"")
    .load(source_csv_path)
)

(
    df_source
    .repartition(1000)
    .write
    .format("json")
    .mode("overwrite")
    .save(landing_path)
)

generated_files = [f for f in dbutils.fs.ls(landing_path) if f.name.endswith(".json")]
print(f"Total files generated: {len(generated_files)}")

### Step 2: Initial Auto Loader Stream Ingestion
* Configures PySpark Structured Streaming with `cloudFiles` format and `availableNow=True` trigger.
* Automatically discovers incoming JSON files, captures schema variations via `schemaHints`, and appends clean records to the Bronze Delta table.

In [0]:

landing_path = "abfss://natalkamartinuk55@dlsua5816bd.dfs.core.windows.net/autoloader_source/initial_files"
schema_path = "abfss://natalkamartinuk55@dlsua5816bd.dfs.core.windows.net/autoloader_source/schema_checkpoint"
checkpoint_path = "abfss://natalkamartinuk55@dlsua5816bd.dfs.core.windows.net/autoloader_source/stream_checkpoint"

target_table = "dbr_dev_ua5816bd.natalkamartinuk55_bronze.autoloader_youtube_bronze"

df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(landing_path)
)

query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:

df_bronze_check = spark.table("dbr_dev_ua5816bd.natalkamartinuk55_bronze.autoloader_youtube_bronze")

print(f"Total rows ingested: {df_bronze_check.count()}")
display(df_bronze_check.limit(5))

### Step 3: Schema Evolution Trigger (Dynamic Schema Change)
* Generates an updated payload containing an unmapped technical column: `stream_source_tag`.
* Injects the record into the landing directory to validate Auto Loader dynamic adaptation.

In [0]:
from pyspark.sql.functions import lit

df_sample = spark.table(target_table).limit(1)

df_evolution_record = (
    df_sample
    .withColumn("video_id", lit("NEW_EVOLVED_1001"))
    .withColumn("title", lit("Auto Loader Schema Evolution Demo"))
    .withColumn("stream_source_tag", lit("EVOLUTION_SUCCESS_TAG"))  
    .drop("_rescued_data")  
)

new_file_landing_path = "abfss://natalkamartinuk55@dlsua5816bd.dfs.core.windows.net/autoloader_source/initial_files/schema_test_record.json"

(
    df_evolution_record
    .coalesce(1)
    .write
    .format("json")
    .mode("overwrite")
    .save(new_file_landing_path)
)

### Step 5: Dynamic Schema Evolution Execution
* Re-executes the Auto Loader ingestion stream with `cloudFiles.schemaEvolutionMode = "addNewColumns"`.
* Delta Lake dynamically migrates the table schema via `option("mergeSchema", "true")` without schema lockouts or stream failure.

In [0]:

dbutils.fs.rm(checkpoint_path, recurse=True)

df_stream_evolved = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.schemaHints", "comments_disabled BOOLEAN, ratings_disabled BOOLEAN, video_error_or_removed BOOLEAN, views LONG, likes LONG, dislikes LONG, comment_count LONG, category_id LONG")
    .load(landing_path)
)

query_evolution = (
    df_stream_evolved.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(target_table)
)

query_evolution.awaitTermination()

### Step 6: Schema Evolution Verification
* Queries the Delta table filtering on `stream_source_tag IS NOT NULL`.
* Confirms the new column has been added to the physical metadata and populated with valid payloads.

In [0]:

df_final_check = spark.table("dbr_dev_ua5816bd.natalkamartinuk55_bronze.autoloader_youtube_bronze")

print(f"Total rows in Bronze: {df_final_check.count()}")
display(
    df_final_check
    .filter("stream_source_tag IS NOT NULL")
    .select("video_id", "title", "stream_source_tag", "_rescued_data")
)

### Step 7: Checkpoint Validation & Stream Idempotency Test
* Triggers the streaming job when no new files are present in the landing location.
* Verifies that the committed checkpoint offset processes 0 rows, guaranteeing idempotent behavior and zero data duplication.

In [0]:

query_noop = (
    df_stream_evolved.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query_noop.awaitTermination()

last_progress = query_noop.lastProgress
num_input_rows = last_progress["numInputRows"] if last_progress else 0

print(f"Rows processed in this batch: {num_input_rows} (Expected: 0)")